In [1]:
!pip install langchain_huggingface

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_huggingface.llms import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace

In [2]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-Coder-Next",
    task='text-generation'
)

model=ChatHuggingFace(llm=llm)


In [ ]:
import os
hf_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")


In [ ]:
# tool creation

from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str , target_currency:str) -> float:
  """This function fetches the currency conversion factor between the given base currency and target currency"""
  url=f'https://v6.exchangerate-api.com/v6/_______/pair/{base_currency}/{target_currency}'

  response=requests.get(url)
  return response.json()


@tool
def convert(base_currency_value : int , conversion_factor : Annotated[float , InjectedToolArg]) -> float:
  """
    MUST be used for ALL currency multiplication.
    Do NOT calculate manually.
    Always use this tool for computing final conversion amount.: Given a currency conversion rate , this function calculates the target currency value from a given base currency value"""
  return base_currency_value * conversion_factor

In [51]:
get_conversion_factor.invoke({'base_currency':'USD' , 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1771891201,
 'time_last_update_utc': 'Tue, 24 Feb 2026 00:00:01 +0000',
 'time_next_update_unix': 1771977601,
 'time_next_update_utc': 'Wed, 25 Feb 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.9674}

In [52]:
convert.invoke({ 'base_currency_value' : 10, 'conversion_factor' : 90.9674})

909.674

In [53]:
# tool binding
llm_with_tools=model.bind_tools([get_conversion_factor , convert])

In [66]:
# tool calling
messages=[(HumanMessage('"First, get the conversion factor between USD and INR. Then, use that factor to convert 10 USD to INR."'))]

In [67]:
ai_message=llm_with_tools.invoke(messages)

In [68]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_73d69d8b502e40aaa16b80b3',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_45630b5c1ed54c38856989e1',
  'type': 'tool_call'}]

In [69]:
messages.append(ai_message)

In [70]:
import json
for tool_call in ai_message.tool_calls:
  # execute the first tool
  if tool_call['name']=='get_conversion_factor':
    tool_message1=get_conversion_factor.invoke(tool_call)
    conversion_rate=json.loads(tool_message1.content)["conversion_rate"]  #used json.loads because the result was in json format , so we converted it to a python dict
    messages.append(tool_message1)
  # execute the second tool
  elif tool_call['name']=='convert':
    #fetching the args of convert and appending conversion rate in it
    tool_call['args']['conversion_factor']=conversion_rate
    tool_message2=convert.invoke(tool_call)
    messages.append(tool_message2)

In [71]:
messages

[HumanMessage(content='"First, get the conversion factor between USD and INR. Then, use that factor to convert 10 USD to INR."', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_73d69d8b502e40aaa16b80b3', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_45630b5c1ed54c38856989e1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 454, 'total_tokens': 514}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8faf-4f00-7c20-9c8a-a23869d9874d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_73d69d8b502e40aaa16b80

In [75]:
llm_with_tools.invoke(messages).content

'The conversion factor from USD to INR is **90.9674**. Using this factor, 10 USD is equivalent to **909.674 INR**.'